<a href="https://colab.research.google.com/github/Heptazero/nn-labs/blob/main/associative-memory/experiments/modern-hopfield-playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 下载

# New Section

In [ ]:
!pip install -q entmax
from entmax import sparsemax

import torch
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

# transforms.Normalize(0.5, 0.5) 做的事：(像素值 - 0.5) / 0.5
# 把 [0,1] 的灰度值搬到 [-1,1]，这是论文里的约定
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.5, 0.5),
])

dataset = torchvision.datasets.MNIST(
    root="./mnist_data", train=True, download=True, transform=transform
)

image, label = dataset[0]  # 取第 0 张图
print("张量形状：", image.shape)
print("像素值范围：", image.min().item(), "到", image.max().item())
print("标签：", label)

plt.imshow(image.squeeze(), cmap="gray", vmin=-1, vmax=1)
plt.title(f"label = {label}")
plt.show()

In [ ]:
loader = torch.utils.data.DataLoader(dataset, batch_size=5000, shuffle=True)
images, labels = next(iter(loader))

print("一批的形状：", images.shape)

images_flat = images.reshape(images.shape[0], -1)
print("拉直后的形状：", images_flat.shape)

In [ ]:
def mask_rows(image, perc):
    """把图像最后 perc 比例的行整体置 0，模拟这部分记忆丢失"""
    image = image.clone()
    rows = image.shape[-2]  # 28
    rows_to_zero = int(perc * rows)
    image[..., -rows_to_zero:, :] = 0
    return image

target = images[0]              # 取一张图当作"记忆"
cue = mask_rows(target, perc=0.5)  # 遮住下半部分（perc=0.5 表示遮一半）

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(target.squeeze(), cmap="gray", vmin=-1, vmax=1)
axes[0].set_title("原始记忆")
axes[0].axis("off")
axes[1].imshow(cue.squeeze(), cmap="gray", vmin=-1, vmax=1)
axes[1].set_title("查询（遮挡下半）")
axes[1].axis("off")
plt.show()

In [ ]:
def add_gaussian_noise(image, std, seed=0):
    """给图像加高斯噪声，然后裁剪回 [-1,1]"""
    generator = torch.Generator().manual_seed(seed)
    noise = torch.randn(image.shape, generator=generator) * std
    return torch.clamp(image + noise, -1.0, 1.0)

noisy = add_gaussian_noise(target, std=0.5)
cue = mask_rows(noisy, perc=0.5)  # 先加噪声，再遮挡——遮挡区域会变回精确的 0，不带噪声

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(target.squeeze(), cmap="gray", vmin=-1, vmax=1)
axes[0].set_title("原始记忆")
axes[0].axis("off")
axes[1].imshow(noisy.squeeze(), cmap="gray", vmin=-1, vmax=1)
axes[1].set_title("加噪声后")
axes[1].axis("off")
axes[2].imshow(cue.squeeze(), cmap="gray", vmin=-1, vmax=1)
axes[2].set_title("查询（噪声+遮挡）")
axes[2].axis("off")
plt.show()

In [ ]:
import torch.nn.functional as F

beta = 8.0
memories = images_flat[:100]   # 先存 100 条记忆
query = cue.flatten()          # 查询也要拉直成 784 维，和记忆形状对齐

scores = beta * (memories @ query)      # 每条记忆和查询的内积（相似度），乘 β 缩放
weights = torch.softmax(scores, dim=0)  # 归一化成一个概率分布：越像查询，权重越大
output = weights @ memories             # 用这组权重对所有记忆做加权平均

similarity = F.cosine_similarity(output.unsqueeze(0), target.flatten().unsqueeze(0)).item()
print("检索结果和原图的余弦相似度：", similarity)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img, title in zip(
    axes,
    [target.squeeze(), cue.squeeze(), output.reshape(28, 28)],
    ["原始记忆", "查询", "检索结果"],
):
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.show()

In [ ]:
beta = 0.2
scores = beta * (memories @ query)

weights = torch.softmax(scores, dim=0)
weights_entmax = sparsemax(scores, dim=0)

output = weights @ memories
output_entmax = weights_entmax @ memories

print("softmax   相似度：", F.cosine_similarity(output.unsqueeze(0), target.flatten().unsqueeze(0)).item())
print("sparsemax 相似度：", F.cosine_similarity(output_entmax.unsqueeze(0), target.flatten().unsqueeze(0)).item())
print("softmax   非零权重：", (weights > 1e-6).sum().item(), "/ 100")
print("sparsemax 非零权重：", (weights_entmax > 1e-6).sum().item(), "/ 100")

In [ ]:
beta = 0.2
memories = images_flat[:2000]
scores = beta * (memories @ query)

weights = torch.softmax(scores, dim=0)
weights_entmax = sparsemax(scores, dim=0)

output = weights @ memories
output_entmax = weights_entmax @ memories

print("softmax   相似度：", F.cosine_similarity(output.unsqueeze(0), target.flatten().unsqueeze(0)).item())
print("sparsemax 相似度：", F.cosine_similarity(output_entmax.unsqueeze(0), target.flatten().unsqueeze(0)).item())
print("softmax   非零权重：", (weights > 1e-6).sum().item(), "/ 2000")
print("sparsemax 非零权重：", (weights_entmax > 1e-6).sum().item(), "/ 2000")

In [ ]:
theta = beta * (memories @ query)   # 已经乘过 β 的分数，就是 sparsemax 输入的那个 θ
top2 = torch.topk(theta, 2).values
gap = (top2[0] - top2[1]).item()

margin_sparsemax = 1.0   # α=2 的 margin

print("最高分与次高分的差距：", gap)
print("sparsemax 的 margin：   ", margin_sparsemax)
print("差距是否超过 margin（预测会精确锁定单条记忆）：", gap > margin_sparsemax)

In [ ]:
cue_hard = mask_rows(add_gaussian_noise(target, std=0.5), perc=0.9)  # 遮挡 90%，只留很小一条线索
query_hard = cue_hard.flatten()

theta_hard = beta * (memories @ query_hard)
top2_hard = torch.topk(theta_hard, 2).values
gap_hard = (top2_hard[0] - top2_hard[1]).item()

weights_entmax_hard = sparsemax(theta_hard, dim=0)
output_entmax_hard = weights_entmax_hard @ memories
similarity_hard = F.cosine_similarity(output_entmax_hard.unsqueeze(0), target.flatten().unsqueeze(0)).item()

print("差距：", gap_hard, "  margin：", margin_sparsemax, "  预测精确锁定：", gap_hard > margin_sparsemax)
print("sparsemax 非零权重数：", (weights_entmax_hard > 1e-6).sum().item())
print("sparsemax 相似度：", similarity_hard)

In [ ]:
def transform(theta, method):
    if method == "softmax":
        return torch.softmax(theta, dim=0)
    elif method == "sparsemax":
        return sparsemax(theta, dim=0)
    else:
        raise ValueError(method)

def success_rate(n, method, beta, perc=0.5, std=0.0, seed=0):
    torch.manual_seed(seed)
    X = images[:n]                                              # 存 n 条记忆
    Q = mask_rows(add_gaussian_noise(X, std=std, seed=seed), perc=perc)  # 每条记忆各自造一条查询

    X_flat = X.reshape(n, -1)
    Q_flat = Q.reshape(n, -1)

    theta = beta * (X_flat @ Q_flat.T)   # (n, n)：第 i 行第 j 列 = 记忆 i 和查询 j 的打分
    weights = transform(theta, method)   # 每一列（每条查询）各自归一化
    output = weights.T @ X_flat          # 每条查询各自的检索结果，(n, d)

    similarities = F.cosine_similarity(output, X_flat, dim=1)  # 第 j 条结果 vs 第 j 条真正的记忆
    return (similarities > 0.9).float().mean().item()          # 多少比例的查询成功找回了自己


print("softmax   成功率 (n=2000)：", success_rate(2000, "softmax", beta=0.1))
print("sparsemax 成功率 (n=2000)：", success_rate(2000, "sparsemax", beta=0.1))

In [ ]:
import numpy as np

memory_sizes = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

def sweep(method, beta):
    medians = []
    for n in memory_sizes:
        rates = [success_rate(n, method, beta, seed=seed) for seed in range(5)]
        medians.append(np.median(rates))
        print(f"n={n:5d}  {method:9s}  median success rate = {medians[-1]:.3f}")
    return medians

softmax_curve_beta1 = sweep("softmax", beta=0.1)

In [ ]:
softmax_curve = sweep("softmax", beta=0.1)
sparsemax_curve = sweep("sparsemax", beta=0.1)

plt.figure(figsize=(7, 4))
plt.plot(memory_sizes, softmax_curve, marker="o", label="softmax (1-entmax)")
plt.plot(memory_sizes, sparsemax_curve, marker="s", label="sparsemax (2-entmax)")
plt.xscale("log", base=2)
plt.xlabel("Number of Memories")
plt.ylabel("Success Retrieval Rate")
plt.ylim(0, 1.05)
plt.legend()
plt.title("MNIST 检索容量曲线（β=0.1）")
plt.grid(True, alpha=0.3)
plt.show()